In [1]:
from pathlib import Path
import re
import pandas as pd

root = Path().resolve()
example_file = next(root.glob("001~050 Cycles/*.xlsx"))

xls = pd.ExcelFile(example_file)
print("Example file:", example_file.name)
print("Sheets:", xls.sheet_names)

# Try to find the statistics-like sheet name robustly
stats_sheet = next((s for s in xls.sheet_names if "stat" in s.lower()), xls.sheet_names[0])
print("Using sheet:", stats_sheet)

df_stats = pd.read_excel(example_file, sheet_name=stats_sheet)
print("Columns:")
print(df_stats.columns.tolist())
print("\nHead:")
print(df_stats.head(5))

Example file: DOE-001-050-10DU 01.xlsx
Sheets: ['Global_Info', 'Statistics_49', 'Channel_49_1', 'Statistics_50', 'Channel_50_1', 'Statistics_51', 'Channel_51_1', 'Statistics_52', 'Channel_52_1', 'Statistics_53', 'Channel_53_1', 'Statistics_54', 'Channel_54_1', 'Statistics_55', 'Channel_55_1', 'Statistics_56', 'Channel_56_1']
Using sheet: Statistics_49
Columns:
['Date_Time', 'Test_Time(s)', 'Step_Time(s)', 'Step_Index', 'Cycle_Index', 'Voltage(V)', 'Current(A)', 'Charge_Capacity(Ah)', 'Discharge_Capacity(Ah)', 'Charge_Energy(Wh)', 'Discharge_Energy(Wh)', 'ACR(Ohm)', 'Internal Resistance(Ohm)']

Head:
            Date_Time  Test_Time(s)  Step_Time(s)  Step_Index  Cycle_Index  \
0 2018-01-27 19:25:36  13166.567720      300.0029           6            1   
1 2018-01-27 21:59:09  22379.581760      300.0017           6            2   
2 2018-01-28 00:33:59  31668.826132      300.0015           6            3   
3 2018-01-28 03:09:22  40991.815300      300.0010           6            4   
4 2

In [2]:
# Inspect metadata sheet structure
meta = pd.read_excel(example_file, sheet_name='Global_Info', header=None)
print(meta.head(30).to_string(index=False, header=False))

Sample No Channel
   01-001      49
   01-002      50
   01-003      51
   01-004      52
   01-005      53
   01-006      54
   01-007      55
   01-008      56


In [3]:
meta_df = pd.read_excel(example_file, sheet_name='Global_Info')
print('Global_Info columns:', meta_df.columns.tolist())
print(meta_df)

# Check if there are additional non-empty cells when reading raw matrix
raw = pd.read_excel(example_file, sheet_name='Global_Info', header=None)
print('Raw shape:', raw.shape)
non_empty = raw.stack(dropna=True).reset_index()
non_empty.columns = ['row', 'col', 'value']
print(non_empty.head(40))

Global_Info columns: ['Sample No', 'Channel']
  Sample No  Channel
0    01-001       49
1    01-002       50
2    01-003       51
3    01-004       52
4    01-005       53
5    01-006       54
6    01-007       55
7    01-008       56
Raw shape: (9, 2)
    row  col      value
0     0    0  Sample No
1     0    1    Channel
2     1    0     01-001
3     1    1         49
4     2    0     01-002
5     2    1         50
6     3    0     01-003
7     3    1         51
8     4    0     01-004
9     4    1         52
10    5    0     01-005
11    5    1         53
12    6    0     01-006
13    6    1         54
14    7    0     01-007
15    7    1         55
16    8    0     01-008
17    8    1         56


C:\Users\RXT0TKQ\AppData\Local\Temp\ipykernel_32624\3481366546.py:8: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  non_empty = raw.stack(dropna=True).reset_index()


In [4]:
# Inspect sample ID patterns across a few files
sample_files = sorted(root.glob('*Cycles/*.xlsx'))[:20]
rows = []
for f in sample_files:
    g = pd.read_excel(f, sheet_name='Global_Info')
    cols = [str(c).strip() for c in g.columns]

    # Normalize common variant headers
    sample_col = next((c for c in g.columns if str(c).strip().lower() in {'sample no', 'sample_no', 'sample'}), None)
    if sample_col is None:
        # Fallback: first non-numeric-looking column likely stores sample ids
        object_cols = [c for c in g.columns if g[c].dtype == 'object']
        sample_col = object_cols[0] if object_cols else g.columns[0]

    sample_ids = g[sample_col].astype(str).tolist()
    prefixes = sorted({s.split('-')[0] for s in sample_ids if '-' in s})
    rows.append({
        'file': f.name,
        'folder': f.parent.name,
        'cols': cols,
        'sample_col': str(sample_col),
        'sample_prefixes': prefixes,
        'sample_min': min(sample_ids),
        'sample_max': max(sample_ids),
        'n_rows': len(g),
    })

pd.DataFrame(rows)

,file,folder,cols,sample_col,sample_prefixes,sample_min,sample_max,n_rows
0,DOE-001-050-10DU 01.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[01],01-001,01-008,8
1,DOE-001-050-10DU 02.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[02],02-009,02-016,8
2,DOE-001-050-10DU 03.xlsx,001~050 Cycles,"[Sample No, Channel, Unnamed: 2]",Sample No,[03],03-017,03-024,8
3,DOE-001-050-10DU 04.xlsx,001~050 Cycles,"[Sample No, Channel, Unnamed: 2]",Sample No,[04],04-025,04-032,8
4,DOE-001-050-10DU 05.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[05],05-033,05-040,8
5,DOE-001-050-10DU 06.xlsx,001~050 Cycles,"[Sample No, Channel, Unnamed: 2]",Sample No,[06],06-041,06-048,8
6,DOE-001-050-25DU 07.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[07],07-049,07-056,8
7,DOE-001-050-25DU 08.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[08],08-057,08-064,8
8,DOE-001-050-25DU 09.xlsx,001~050 Cycles,"[Unnamed: 0, Unnamed: 1, Unnamed: 2, Unnamed: ...",Unnamed: 0,[09],09-065,nan,11
9,DOE-001-050-25DU 10.xlsx,001~050 Cycles,"[Sample No, Channel]",Sample No,[10],10-073,10-080,8


In [5]:
# Check raw channel data for temperature/current clues
chan_sheet = next((s for s in xls.sheet_names if s.lower().startswith('channel_')), None)
print('Channel sheet:', chan_sheet)
chan = pd.read_excel(example_file, sheet_name=chan_sheet)
print('Channel columns:', chan.columns.tolist())
print(chan.head(5))

Channel sheet: Channel_49_1
Channel columns: ['Date_Time', 'Test_Time(s)', 'Step_Time(s)', 'Step_Index', 'Cycle_Index', 'Voltage(V)', 'Current(A)', 'Charge_Capacity(Ah)', 'Discharge_Capacity(Ah)', 'Charge_Energy(Wh)', 'Discharge_Energy(Wh)', 'ACR(Ohm)', 'Internal Resistance(Ohm)']
                Date_Time  Test_Time(s)  Step_Time(s)  Step_Index  \
0 2018-01-27 15:48:09.697      120.0002      120.0002           1   
1 2018-01-27 15:50:09.697      240.0001      240.0001           1   
2 2018-01-27 15:52:09.697      360.0003      360.0003           1   
3 2018-01-27 15:54:09.697      480.0005      480.0005           1   
4 2018-01-27 15:56:09.697      600.0005      600.0005           1   

   Cycle_Index  Voltage(V)  Current(A)  Charge_Capacity(Ah)  \
0            1    3.409640         0.0                  0.0   
1            1    3.410280         0.0                  0.0   
2            1    3.410618         0.0                  0.0   
3            1    3.410776         0.0             

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path(r"c:\Users\RXT0TKQ\OneDrive - NEE\Desktop\Dev\Tuggle_ML_Project")
OUT_CSV = ROOT / "battery_stats_cycles_001_750.csv"

folder_pattern = re.compile(r"^(\d{3})~(\d{3})")
sample_id_pattern = re.compile(r"^\d{2}-\d{3}$")

# Optional mapping from experimental sample group (01..24) to metadata.
# Fill these from your DOE reference if you have exact values.
DOE_META_BY_GROUP = {
    # 1: {"temp_c": 25, "charge_cutoff_current_a": 0.2},
    # 2: {"temp_c": 35, "charge_cutoff_current_a": 0.2},
}


def parse_folder_cycle_bounds(folder_name: str):
    m = folder_pattern.search(folder_name)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def parse_discharge_rate(file_name: str):
    m = re.search(r"-(\d+)DU", file_name, flags=re.IGNORECASE)
    return float(m.group(1)) if m else np.nan


def normalize_cycle_col(columns):
    for c in columns:
        if str(c).strip().lower() in {"cycle_index", "cycle index", "cycle"}:
            return c
    for c in columns:
        if "cycle" in str(c).strip().lower():
            return c
    return None


def normalize_voltage_col(columns):
    for c in columns:
        t = str(c).strip().lower()
        if t in {"voltage(v)", "voltage"}:
            return c
    for c in columns:
        if "voltage" in str(c).strip().lower():
            return c
    return None


def normalize_discharge_energy_col(columns):
    for c in columns:
        t = str(c).strip().lower()
        if t in {"discharge_energy(wh)", "discharge energy", "discharge_energy"}:
            return c
    for c in columns:
        if "discharge" in str(c).strip().lower() and "energy" in str(c).strip().lower():
            return c
    return None


def extract_channel_sample_map(xls_path: Path):
    # Read as raw table to handle header inconsistencies.
    raw = pd.read_excel(xls_path, sheet_name="Global_Info", header=None)
    mapping = {}

    for _, row in raw.iterrows():
        vals = [x for x in row.tolist() if pd.notna(x)]
        if len(vals) < 2:
            continue

        sample_candidate = str(vals[0]).strip()
        if not sample_id_pattern.match(sample_candidate):
            continue

        chan = None
        for v in vals[1:]:
            sv = str(v).strip()
            if re.fullmatch(r"\d+", sv):
                chan = int(sv)
                break
            if isinstance(v, (int, float)) and float(v).is_integer():
                chan = int(v)
                break

        if chan is not None:
            mapping[chan] = sample_candidate

    return mapping


def parse_channel_from_sheet(sheet_name: str):
    m = re.search(r"statistics_(\d+)", sheet_name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"statisticsbycycle-chan_(\d+)", sheet_name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


records = []
excel_files = sorted(ROOT.glob("*Cycles/*.xlsx"))

for xlsx in excel_files:
    folder_name = xlsx.parent.name
    bounds = parse_folder_cycle_bounds(folder_name)
    if bounds is None:
        continue

    folder_start, folder_end = bounds
    discharge_rate = parse_discharge_rate(xlsx.name)

    try:
        channel_to_sample = extract_channel_sample_map(xlsx)
    except Exception as e:
        print(f"Skipping {xlsx.name}: failed Global_Info parse ({e})")
        continue

    try:
        xls = pd.ExcelFile(xlsx)
    except Exception as e:
        print(f"Skipping {xlsx.name}: failed to open workbook ({e})")
        continue

    stats_sheets = [
        s for s in xls.sheet_names
        if str(s).lower().startswith("statistics_") or str(s).lower().startswith("statisticsbycycle-")
    ]

    for sheet in stats_sheets:
        chan = parse_channel_from_sheet(sheet)
        sample_no = channel_to_sample.get(chan)
        sample_group = int(sample_no.split("-")[0]) if isinstance(sample_no, str) and "-" in sample_no else np.nan

        meta = DOE_META_BY_GROUP.get(sample_group, {}) if pd.notna(sample_group) else {}
        temp_c = meta.get("temp_c", np.nan)
        charge_cutoff_current_a = meta.get("charge_cutoff_current_a", np.nan)

        try:
            sdf = pd.read_excel(xlsx, sheet_name=sheet)
        except Exception as e:
            print(f"Skipping sheet {xlsx.name}::{sheet} ({e})")
            continue

        cycle_col = normalize_cycle_col(sdf.columns)
        voltage_col = normalize_voltage_col(sdf.columns)
        de_col = normalize_discharge_energy_col(sdf.columns)

        if not (cycle_col and voltage_col and de_col):
            print(f"Skipping sheet {xlsx.name}::{sheet} due to missing required columns")
            continue

        out = pd.DataFrame({
            "cycle_number_local": pd.to_numeric(sdf[cycle_col], errors="coerce"),
            "voltage_v": pd.to_numeric(sdf[voltage_col], errors="coerce"),
            "discharge_energy_wh": pd.to_numeric(sdf[de_col], errors="coerce"),
        })

        out = out.dropna(subset=["cycle_number_local"]).copy()
        out["cycle_number_local"] = out["cycle_number_local"].astype(int)
        out = out[out["cycle_number_local"] >= 1]

        # Map per-folder local cycle index (1..50) to global cycle count (1..750).
        out["cycle_number"] = folder_start + out["cycle_number_local"] - 1

        # Keep only rows in the folder's cycle window.
        out = out[(out["cycle_number"] >= folder_start) & (out["cycle_number"] <= folder_end)]

        out["sample_number"] = sample_no
        out["temp_c"] = temp_c
        out["charge_cutoff_current_a"] = charge_cutoff_current_a
        out["discharge_rate_du"] = discharge_rate

        out["source_file"] = xlsx.name
        out["source_folder"] = folder_name
        out["source_sheet"] = sheet

        records.append(out)

if not records:
    raise RuntimeError("No records were extracted. Check workbook formats and paths.")

final_df = pd.concat(records, ignore_index=True)
final_df = final_df[[
    "cycle_number",
    "voltage_v",
    "sample_number",
    "discharge_energy_wh",
    "temp_c",
    "charge_cutoff_current_a",
    "discharge_rate_du",
    "cycle_number_local",
    "source_folder",
    "source_file",
    "source_sheet",
]].sort_values(["sample_number", "cycle_number"], kind="stable")

final_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")
print(f"Rows: {len(final_df):,}")
print(f"Unique samples: {final_df['sample_number'].nunique()}")
print(f"Cycle range: {final_df['cycle_number'].min()} to {final_df['cycle_number'].max()}")
print(final_df.head(10))

Saved: c:\Users\RXT0TKQ\OneDrive - NEE\Desktop\Dev\Tuggle_ML_Project\battery_stats_cycles_001_750.csv
Rows: 93,467
Unique samples: 196
Cycle range: 1 to 750
   cycle_number  voltage_v sample_number  discharge_energy_wh  temp_c  \
0             1   3.669531        01-001             9.198724     NaN   
1             2   3.664757        01-001             9.472517     NaN   
2             3   3.659098        01-001             9.566450     NaN   
3             4   3.655673        01-001             9.611696     NaN   
4             5   3.653233        01-001             9.635559     NaN   
5             6   3.650719        01-001             9.666832     NaN   
6             7   3.648497        01-001             9.680622     NaN   
7             8   3.645295        01-001             9.698596     NaN   
8             9   3.644960        01-001             9.696089     NaN   
9            10   3.644005        01-001             9.709321     NaN   

   charge_cutoff_current_a  discharge_r

In [7]:
# Validate extraction coverage by folder and cycle range
summary = final_df.groupby('source_folder').agg(
    rows=('cycle_number', 'size'),
    min_cycle=('cycle_number', 'min'),
    max_cycle=('cycle_number', 'max'),
    files=('source_file', 'nunique'),
    samples=('sample_number', 'nunique'),
).reset_index().sort_values('source_folder')

print(summary.to_string(index=False))

  source_folder  rows  min_cycle  max_cycle  files  samples
 001~050 Cycles  8950          1         50     24      179
 051~100 Cycles  8924         51        100     24      179
 101~150 Cycles  9086        101        150     24      178
 151~200 Cycles  8862        151        200     23      168
 201~250 Cycles  8567        201        250     23      171
 251~300 Cycles  7950        251        300     21      159
 301~350 Cycles  7050        301        350     18      141
 351~400 Cycles  7000        351        400     18      140
 401~450 Cycles  7052        401        450     19      133
 451~500 Cycles  6059        451        500     16      122
 501~550 Cycles  5550        501        550     15      111
551~600  Cycles  4200        551        600     11       84
601~650  Cycles  2700        601        650      7       54
651~700  Cycles  1117        651        700      3       23
701~750  Cycles   400        701        750      1        8


In [8]:
# Inspect late-cycle folders to see why they were not included
late_files = sorted((ROOT / '651~700  Cycles').glob('*.xlsx')) + sorted((ROOT / '701~750  Cycles').glob('*.xlsx'))
print('Late files found:', len(late_files))
for f in late_files[:6]:
    try:
        x = pd.ExcelFile(f)
        stats = [s for s in x.sheet_names if str(s).lower().startswith('statistics_')]
        print(f.name, '| sheets:', len(x.sheet_names), '| stats_sheets:', len(stats), '| first sheets:', x.sheet_names[:5])
    except Exception as e:
        print(f.name, '| error opening:', e)

Late files found: 4
DOE-651-700-10DU-03.xlsx | sheets: 15 | stats_sheets: 0 | first sheets: ['Global_Info', 'StatisticsByCycle-Chan_65', 'StatisticsByStep-Chan_65', 'StatisticsByCycle-Chan_66', 'StatisticsByStep-Chan_66']
DOE-651-700-10DU-05.xlsx | sheets: 41 | stats_sheets: 0 | first sheets: ['Global_Info', 'RawData_81_1', 'Channel_81_1', 'Chart_Channel_81', 'StatisticsByCycle-Chan_81']
DOE-651-700-45DU-17.xlsx | sheets: 41 | stats_sheets: 0 | first sheets: ['Global_Info', 'RawData_215_1', 'Channel_215_1', 'Chart_Channel_215', 'StatisticsByCycle-Chan_215']
DOE-701-750-10DU-05.xlsx | sheets: 41 | stats_sheets: 0 | first sheets: ['Global_Info', 'RawData_80_1', 'Channel_80_1', 'Chart_Channel_80', 'StatisticsByCycle-Chan_80']


In [9]:
# Inspect one StatisticsByCycle sheet schema
lf = late_files[0]
x = pd.ExcelFile(lf)
bycycle = [s for s in x.sheet_names if 'statisticsbycycle' in s.lower()]
print('File:', lf.name)
print('ByCycle sheets:', bycycle[:3])
if bycycle:
    bdf = pd.read_excel(lf, sheet_name=bycycle[0])
    print('Columns:', bdf.columns.tolist())
    print(bdf.head(5))

File: DOE-651-700-10DU-03.xlsx
ByCycle sheets: ['StatisticsByCycle-Chan_65', 'StatisticsByCycle-Chan_66', 'StatisticsByCycle-Chan_67']
Columns: ['Date_Time', 'Test_Time(s)', 'Step_Time(s)', 'Step_Index', 'Cycle_Index', 'Voltage(V)', 'Current(A)', 'Charge_Capacity(Ah)', 'Discharge_Capacity(Ah)', 'Charge_Time(s)', 'DisCharge_Time(s)', 'Charge_Energy(Wh)', 'Discharge_Energy(Wh)', 'Internal Resistance(Ohm)', 'dV/dt(V/s)']
                Date_Time  Test_Time(s)  Step_Time(s)  Step_Index  \
0 2018-06-22 00:26:43.052  14930.826712      300.0045           6   
1 2018-06-22 02:25:24.871  22052.645984      300.0032           6   
2 2018-06-22 04:24:52.348  29220.122496      300.0040           6   
3 2018-06-22 06:24:10.803  36378.579816      300.0050           6   
4 2018-06-22 08:24:20.110  43587.886576      300.0035           6   

   Cycle_Index  Voltage(V)  Current(A)  Charge_Capacity(Ah)  \
0            1    3.705683         0.0             2.527135   
1            2    3.703780         0.